# AutoML(FLAML) 비교 실험

| 항목 | 내용 |
|---|---|
| 종류 | **지도학습 (분류)** |
| 정답(라벨) | 24시간 안에 **압력 저하**(FlowCool Pressure Dropped Below Limit) 고장이 나는가 (예=1) |
| 데이터 | `data/processed/step/` (레시피 스텝 구간 하나 = 한 행), 셔터 1(식각 중)만 |
| 입력 | 핵심(냉각 압력·냉각 유량) + 보조(억제 전류·챔버 압력) 4개 센서의 z값 |
| 질문 | 모델과 설정을 자동으로 찾아도(AutoML) 처음 보는 장비에서 성능이 오르는가? |

## 1. 방법

- **z값 기준**: 각 장비 자신의 **처음 60일** 평소 구간에서 레시피·스텝별 평균·표준편차를 구한다(`self_baseline.ipynb`의 B 방식, 실제 운영에서 가능한 방식). 평소 = 가장 가까운 고장에서 24시간 넘게 떨어진 구간.
- **평가 구간**: 각 장비의 60일 이후. 기록 끝 24시간 안에서 압력 저하 TTF가 비어 판단할 수 없는 구간은 뺀다.
- **입력 20개**: 센서 4개 × (평균·최소·최대) z값 12개 + 장비별 지난 6시간·24시간 동안의 평균 z값 8개.
- **검증 (바깥)**: 지금까지와 같은 장비 단위 4묶음. 묶음마다 5대는 학습·탐색에 전혀 쓰지 않고 채점에만 쓴다.
- **AutoML 탐색 (안쪽)**: FLAML이 학습 장비 15대 안에서 다시 **장비 단위 3묶음 교차검증**(`split_type='group'`)으로 모델과 설정을 고른다. 행을 무작위로 나누면 같은 장비가 섞여 점수가 부풀려지기 때문이다.
- **후보 모델**: LightGBM, XGBoost, RandomForest, ExtraTrees. 묶음당 탐색 시간 5분. 평가 지표 AUC.
- **비교 기준**: 같은 입력에 LightGBM 기본 설정(직접 고른 설정) 하나.

In [1]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import lightgbm as lgb
from flaml import AutoML
from sklearn.metrics import roc_auc_score
from IPython.display import display
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, cwd.parent) if (p / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=480):
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=60))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE', 'ETCHSUPPRESSORCURRENT', 'IONGAUGEPRESSURE']
NAMES = {'FLOWCOOLPRESSURE': '냉각 압력', 'FLOWCOOLFLOWRATE': '냉각 유량',
         'ETCHSUPPRESSORCURRENT': '억제 전류', 'IONGAUGEPRESSURE': '챔버 압력'}
STATS = ['mean', 'min', 'max']
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
TTF = ['TTF_' + f for f in FAULTS]
TTF_DROP = TTF[0]
HOUR, DAY = 3600, 86400
BASE_DAYS = 60
W = 24
BUDGET = 300   # 묶음당 AutoML 탐색 시간(초)
SEED = 0

cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', 'shutter', *TTF,
        *[f'{s}_{st}' for s in SENSORS for st in STATS]]
d = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted((PROC / 'step').glob('*.parquet'))], ignore_index=True)
d['Tool'] = d.Tool.astype(str)

# 평소 여부(가장 가까운 고장까지 24시간 초과)는 고장 3종 모두 기준, 셔터와 무관하게 전체에서 계산
faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.map(lambda t: f'{t[2:]}_{t[:2]}')
fdist = np.full(len(d), np.inf)
for tool, idx in d.groupby('Tool').indices.items():
    ft = np.sort(faults.loc[faults.Tool == tool, 'time'].unique())
    ts, te = d.time_start.to_numpy()[idx], d.time_end.to_numpy()[idx]
    j = np.searchsorted(ft, ts); k = np.searchsorted(ft, te)
    prev = np.where(j > 0, ts - ft[np.maximum(j - 1, 0)], np.inf)
    nxt = np.where(k < len(ft), ft[np.minimum(k, len(ft) - 1)] - te, np.inf)
    fdist[idx] = np.minimum(prev, nxt)
d['normal'] = fdist > DAY
end_t = d.groupby('Tool').time_end.transform('max')
d['censored'] = ((end_t - d.time_end) < DAY) & d[TTF_DROP].isna()
d['start_t'] = d.groupby('Tool').time_start.transform('min')
d['in_base'] = d.time_start < d.start_t + BASE_DAYS * DAY

d = d[d.shutter == 1].copy()
d['_rs'] = d.recipe_step.fillna(-1)
d = d.sort_values(['Tool', 'time_start']).reset_index(drop=True)
y_all = (d[TTF_DROP] <= W * HOUR).to_numpy().astype(int)
print(f'셔터 1 스텝 구간 {len(d):,}행 · 기준 기간(처음 {BASE_DAYS}일) {d.in_base.mean():.1%} · 판단 불가 {d.censored.sum():,}행')

셔터 1 스텝 구간 526,266행 · 기준 기간(처음 60일) 14.3% · 판단 불가 1,255행


## 2. z값 만들기 (장비 자신의 처음 60일 기준)

In [2]:
MEANS = [s + '_mean' for s in SENSORS]
keys = ['Tool', 'recipe', '_rs']
ref = d[d.normal & ~d.censored & d.in_base]
g = ref.groupby(keys)[MEANS]
n = g.size()
mu, sd = g.mean(), g.std()
sd = sd.where((sd > 1e-6) & np.broadcast_to((n >= 10).to_numpy()[:, None], sd.shape))
key = pd.MultiIndex.from_frame(d[keys])
m = mu.reindex(key).to_numpy(); s_ = sd.reindex(key).to_numpy()
Z = {}
for k, s in enumerate(SENSORS):
    for st in STATS:
        Z[f'{s}_{st}_z'] = np.clip((d[f'{s}_{st}'].to_numpy() - m[:, k]) / s_[:, k], -10, 10)
Z = pd.DataFrame(Z).astype('float32')

zm = Z[[f'{s}_mean_z' for s in SENSORS]].copy()
zm.index = pd.to_datetime(d.time_start, unit='s')
parts = []
for win in ['6h', '24h']:
    r = zm.groupby(d.Tool.to_numpy()).rolling(win, min_periods=3).mean().reset_index(level=0, drop=True)
    r.columns = [c + f'_r{win}' for c in zm.columns]
    parts.append(r.reset_index(drop=True))
X = pd.concat([Z] + parts, axis=1).astype('float32')   # d가 장비·시각 순으로 정렬돼 있어 행 순서가 같다
FEATS = list(X.columns)

ev_mask = (~d.in_base & ~d.censored).to_numpy()
print(f'입력 {len(FEATS)}개 · 평가 구간 {ev_mask.sum():,}행 · 고장 전(1) 비율 {y_all[ev_mask].mean():.2%}'
      f' · 냉각 압력 z 있음 {X.loc[ev_mask, "FLOWCOOLPRESSURE_mean_z"].notna().mean():.1%}')

입력 20개 · 평가 구간 449,862행 · 고장 전(1) 비율 6.05% · 냉각 압력 z 있음 95.6%


## 3. 학습과 평가 (장비 단위 4묶음)

In [3]:
tools = sorted(d.Tool.unique(), key=lambda t: f'{t[4:]}{t[:3]}')
perm = np.random.default_rng(SEED).permutation(len(tools))
folds = [[tools[i] for i in perm[k::4]] for k in range(4)]

def make_gbm():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=200,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, is_unbalance=True,
                              random_state=SEED, verbose=-1)

rows, picks, trials = [], [], []
scores = {'LightGBM 기본 설정': np.full(len(d), np.nan), 'AutoML (FLAML)': np.full(len(d), np.nan)}
for k, test_tools in enumerate(folds):
    is_test = d.Tool.isin(test_tools).to_numpy()
    tr, te = ev_mask & ~is_test, ev_mask & is_test
    Xtr, ytr, Xte, yte = X[tr], y_all[tr], X[te], y_all[te]

    t0 = time.time()
    base = make_gbm().fit(Xtr, ytr)
    scores['LightGBM 기본 설정'][te] = base.predict_proba(Xte)[:, 1]
    rows.append({'모델': 'LightGBM 기본 설정', '묶음': k + 1, 'AUC': roc_auc_score(yte, scores['LightGBM 기본 설정'][te]),
                 '시간(초)': round(time.time() - t0)})

    t0 = time.time()
    automl = AutoML()
    automl.fit(Xtr, ytr, task='classification', metric='roc_auc', eval_method='cv', n_splits=3,
               split_type='group', groups=d.Tool.to_numpy()[tr], time_budget=BUDGET,
               estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree'], seed=SEED, verbose=0)
    scores['AutoML (FLAML)'][te] = automl.predict_proba(Xte)[:, 1]
    rows.append({'모델': 'AutoML (FLAML)', '묶음': k + 1, 'AUC': roc_auc_score(yte, scores['AutoML (FLAML)'][te]),
                 '시간(초)': round(time.time() - t0)})
    picks.append({'묶음': k + 1, '고른 모델': automl.best_estimator,
                  '탐색 중 AUC (학습 장비 안 3묶음)': round(1 - automl.best_loss, 3),
                  '설정': {a: (round(float(v), 4) if isinstance(v, float) else v) for a, v in automl.best_config.items()}})
    hist = pd.Series([v[0] for v in automl.config_history.values()]).value_counts()
    trials.append(hist.rename(f'묶음 {k + 1}'))
    print(f'묶음 {k + 1} 완료 · 테스트 장비 {test_tools}', flush=True)
res = pd.DataFrame(rows)

묶음 1 완료 · 테스트 장비 ['M01_03', 'M02_07', 'M01_06', 'M02_04', 'M01_08']


묶음 2 완료 · 테스트 장비 ['M02_10', 'M01_09', 'M01_05', 'M02_03', 'M02_05']


묶음 3 완료 · 테스트 장비 ['M01_04', 'M02_02', 'M01_01', 'M01_10', 'M02_01']


묶음 4 완료 · 테스트 장비 ['M01_02', 'M02_06', 'M01_07', 'M02_09', 'M02_08']


### 3-1. AUC 비교

0.5는 무작위다. 묶음별 AUC는 학습·탐색에 전혀 쓰지 않은 장비 5대에서 잰 값이다.

In [4]:
def tool_auc(name):
    sc = scores[name]; out = {}
    for t, idx in d.groupby('Tool').indices.items():
        mm = idx[ev_mask[idx]]
        if len(np.unique(y_all[mm])) == 2:
            out[t] = roc_auc_score(y_all[mm], sc[mm])
    return pd.Series(out)

summ = []
for name in scores:
    r = res[res.모델 == name]; ta = tool_auc(name)
    summ.append({'모델': name, **{f'묶음 {k}': round(v, 3) for k, v in zip(r.묶음, r.AUC)},
                 '평균 AUC': round(r.AUC.mean(), 3), '장비별 중앙값': round(ta.median(), 3),
                 'AUC>0.5 장비': f'{int((ta > .5).sum())}/{len(ta)}', '총 시간(초)': int(r['시간(초)'].sum())})
display(pd.DataFrame(summ))

fig = go.Figure()
for name, color in zip(scores, ['#9aa3ab', '#2a78d6']):
    ta = tool_auc(name)
    fig.add_bar(x=ta.index, y=ta.values, name=name, marker_color=color)
fig.add_hline(y=0.5, line_dash='dash', annotation_text='무작위(0.5)')
fig.update_layout(title='그림 3-1. 장비별 AUC (압력 저하 24시간 전, 학습에 쓰지 않은 장비)', barmode='group', yaxis_title='AUC')
show(fig, 460)

,모델,묶음 1,묶음 2,묶음 3,묶음 4,평균 AUC,장비별 중앙값,AUC>0.5 장비,총 시간(초)
0,LightGBM 기본 설정,0.469,0.445,0.754,0.485,0.538,0.567,13/19,10
1,AutoML (FLAML),0.509,0.715,0.766,0.413,0.601,0.515,13/19,1200


### 3-2. AutoML이 고른 모델

'탐색 중 AUC'는 학습 장비 15대 안에서 장비 단위로 나눠 잰 값이다. 이 값과 실제 테스트 AUC의 차이가 크면, 탐색이 학습 장비에 맞춰졌다는 뜻이다.

In [5]:
display(pd.DataFrame(picks))
print('모델별 시도 횟수')
display(pd.concat(trials, axis=1).fillna(0).astype(int))

,묶음,고른 모델,탐색 중 AUC (학습 장비 안 3묶음),설정
0,1,xgboost,0.750,"{'n_estimators': 4, 'max_leaves': 4, 'min_chil..."
1,2,extra_tree,0.613,"{'n_estimators': 9, 'max_features': 0.3768, 'm..."
2,3,lgbm,0.659,"{'n_estimators': 25, 'num_leaves': 573, 'min_c..."
3,4,lgbm,0.703,"{'n_estimators': 10, 'num_leaves': 50, 'min_ch..."


모델별 시도 횟수


,묶음 1,묶음 2,묶음 3,묶음 4
xgboost,6,0,3,2
lgbm,1,2,12,5
extra_tree,0,3,0,0
rf,0,1,0,2


## 4. 해석과 한계

**결론: AutoML이 확실히 낫다고 보기 어렵다.** 평균 AUC는 올랐지만, 장비별로 보면 오히려 내려갔고 묶음마다 결과가 크게 흔들린다.

| 지표 | LightGBM 기본 | AutoML (FLAML) |
|---|---|---|
| 묶음 평균 AUC | 0.538 | 0.601 |
| 장비별 AUC 중앙값 | **0.567** | 0.515 |
| AUC > 0.5 장비 | 13/19 | 13/19 |
| 묶음별 범위 | 0.445~0.754 | 0.413~0.766 |
| 시간 | 10초 | 1,200초 |

- **평균이 오른 것은 묶음 2 하나 때문이다** (0.445 → 0.715). 묶음 4는 오히려 내려갔다(0.485 → 0.413). 묶음 4개로는 이 정도 차이를 우연과 구분하기 어렵다.
- **탐색 점수가 실제보다 높다.** 학습 장비 안에서 잰 탐색 AUC는 0.61~0.75였지만, 처음 보는 장비에서는 0.41~0.77이었다. 장비 15대 안에서 고른 설정이 다른 장비로 그대로 이어지지 않는다.
- **AutoML은 아주 작은 모델을 골랐다** (트리 4~25개). 복잡하게 만들수록 탐색 점수가 떨어졌다는 뜻이고, 데이터에서 일반화되는 신호가 약하다는 지금까지의 결론과 같다.
- **5분 동안 시도는 묶음당 4~12번뿐이었다** (데이터 약 34만 행 × 장비 단위 3묶음 교차검증). 시간을 늘리면 더 찾을 수 있지만, 위 이유로 큰 개선은 기대하기 어렵다.

**정리**: 모델 선택이 병목이 아니다. 성능을 올리려면 모델보다 입력(장비별 기준, 고장 종류별 분리)과 평가 방식(장비별 경보 기준, 사건 단위 평가)을 바꿔야 한다.
